# 01 Data cleaning

**In:** the raw CSVs `orders.csv`, `orderlines.csv` and `products.csv` (folder `data/raw-data`).
**Out:** `orders_cl.csv`, `orderlines_cl.csv` and `products_cl.csv` (folder `data/clean-data`).

This notebook fixes technical problems inside each table: duplicates, missing values, wrong data types and broken prices.
It does not yet check whether the tables make business sense together. That happens in `02_data-quality.ipynb`.

`brands.csv` needs no cleaning and is used as it is.

## Setup

In [4]:
import pandas as pd

In [5]:
# show numbers with 2 decimals, and allow long tables
pd.set_option("display.float_format", lambda x: "%.2f" % x)
pd.set_option("display.max_rows", 1000)

In [6]:
orders = pd.read_csv("https://raw.githubusercontent.com/AliHaydar2023/eniac-discount-analysis/main/data/raw-data/orders.csv")
orderlines = pd.read_csv("https://raw.githubusercontent.com/AliHaydar2023/eniac-discount-analysis/main/data/raw-data/orderlines.csv")
products = pd.read_csv("https://raw.githubusercontent.com/AliHaydar2023/eniac-discount-analysis/main/data/raw-data/products.csv")

In [7]:
# size of each raw table (rows, columns)
orders.shape, orderlines.shape, products.shape

((226909, 4), (293983, 7), (19326, 7))

## Step 1: Duplicates

In [8]:
# step 1 starts with fresh copies, so the raw tables stay untouched
orders_s1 = orders.copy()
orderlines_s1 = orderlines.copy()
products_s1 = products.copy()

In [9]:
# number of fully duplicated rows in each table
orders_s1.duplicated().sum(), orderlines_s1.duplicated().sum(), products_s1.duplicated().sum()

(np.int64(0), np.int64(0), np.int64(8746))

In [10]:
# remove the duplicated products, keeping one copy of each
products_s1 = products_s1.drop_duplicates()
products_s1.shape

(10580, 7)

**What we see:**
- Orders and orderlines have no duplicated rows.
- Products had 8,746 exact duplicates (45% of the file). After removing them, 10,580 products are left.

## Step 2: Missing values

In [11]:
orders_s2 = orders_s1.copy()
orderlines_s2 = orderlines_s1.copy()
products_s2 = products_s1.copy()

### Orders

In [12]:
# missing values per column
orders_s2.isna().sum()

,0
order_id,0
created_date,0
total_paid,5
state,0


In [13]:
# the orders with no total_paid
orders_s2.loc[orders_s2["total_paid"].isna(), :]

,order_id,created_date,total_paid,state
127701,427314,2017-11-20 18:54:39,NaN,Pending
132013,431655,2017-11-22 12:15:24,NaN,Pending
147316,447411,2017-11-27 10:32:37,NaN,Pending
148833,448966,2017-11-27 18:54:15,NaN,Pending
149434,449596,2017-11-27 21:52:08,NaN,Pending


In [14]:
# remove these 5 orders
orders_s2 = orders_s2.dropna()
orders_s2.shape

(226904, 4)

**What we see:**
- 5 orders have no `total_paid`. That is 0.002% of all orders, so removing them changes nothing for the analysis.
- 226,904 orders are left.

### Orderlines

In [15]:
# missing values per column
orderlines_s2.isna().sum()

,0
id,0
id_order,0
product_id,0
product_quantity,0
sku,0
unit_price,0
date,0


**What we see:** orderlines has no missing values.

### Products

In [16]:
# missing values per column
products_s2.isna().sum()

,0
sku,0
name,0
desc,7
price,46
promo_price,0
in_stock,0
type,50


In [17]:
# the 7 products with no description
products_s2.loc[products_s2["desc"].isna(), ["sku", "name", "desc"]]

,sku,name,desc
16126,WDT0211-A,"Open - Purple 2TB WD 35 ""PC Security Mac hard ...",NaN
16128,APP1622-A,Open - Apple Smart Keyboard Pro Keyboard Folio...,NaN
17843,PAC2334,Synology DS718 + NAS Server | 10GB RAM,NaN
18152,KAN0034-A,Open - Kanex USB-C Gigabit Ethernet Adapter Ma...,NaN
18490,HTE0025,Hyper Pearl 1600mAh battery Mini USB Mirror an...,NaN
18612,OTT0200,OtterBox External Battery Power Pack 20000 mAHr,NaN
18690,HOW0001-A,Open - Honeywell thermostat Lyric zonificador ...,NaN


In [18]:
# the name describes the product, so copy it into the empty descriptions
products_s2.loc[products_s2["desc"].isna(), "desc"] = products_s2.loc[products_s2["desc"].isna(), "name"]
products_s2["desc"].isna().sum()

np.int64(0)

In [19]:
# remove the 46 products with no price: without a price they can't be used for discounts
products_s2 = products_s2.dropna(subset=["price"])
products_s2.shape

(10534, 7)

In [20]:
# missing values left
products_s2.isna().sum()

,0
sku,0
name,0
desc,0
price,0
promo_price,0
in_stock,0
type,50


**What we see:**
- 7 products had no description. Their name now fills the gap.
- 46 products had no price and are removed: 10,534 products are left.
- 50 products have no `type`. They are **kept**: the price and name are fine, and `type` is not needed for cleaning. The product categories notebook (`03_product-categories.ipynb`) sorts products by name as well, so these can still get a category.

## Step 3: Data types and broken prices

In [21]:
orders_s3 = orders_s2.copy()
orderlines_s3 = orderlines_s2.copy()
products_s3 = products_s2.copy()

In [22]:
# data type of every column
orders_s3.dtypes, orderlines_s3.dtypes, products_s3.dtypes

(order_id          int64
 created_date     object
 total_paid      float64
 state            object
 dtype: object,
 id                   int64
 id_order             int64
 product_id           int64
 product_quantity     int64
 sku                 object
 unit_price          object
 date                object
 dtype: object,
 sku            object
 name           object
 desc           object
 price          object
 promo_price    object
 in_stock        int64
 type           object
 dtype: object)

**What we see:**
- `created_date` (orders) and `date` (orderlines) are stored as text (`object`). They need to be dates.
- `unit_price` (orderlines) and `price` (products) are stored as text. They need to be numbers.

### Dates

In [23]:
# turn the text dates into real dates
orders_s3["created_date"] = pd.to_datetime(orders_s3["created_date"])
orderlines_s3["date"] = pd.to_datetime(orderlines_s3["date"])
orders_s3["created_date"].dtype, orderlines_s3["date"].dtype

(dtype('<M8[ns]'), dtype('<M8[ns]'))

In [24]:
# first and last date in each table
orders_s3["created_date"].min(), orders_s3["created_date"].max(), orderlines_s3["date"].min(), orderlines_s3["date"].max()

(Timestamp('2017-01-01 00:07:19'),
 Timestamp('2018-03-14 13:58:36'),
 Timestamp('2017-01-01 00:07:19'),
 Timestamp('2018-03-14 13:58:36'))

**What we see:** both date columns are now real dates. Both tables run from 1 January 2017 to 14 March 2018.

### Orderlines: `unit_price`

In [25]:
# how many dots each unit_price has (a normal price has exactly 1)
orderlines_s3["unit_price"].str.count(r"\.").value_counts()

,count
unit_price,
1,257814
2,36169


In [26]:
# a few of the prices with 2 dots
orderlines_s3.loc[orderlines_s3["unit_price"].str.count(r"\.") > 1, "unit_price"].head()

,unit_price
6,1.137.99
11,2.565.99
15,3.278.99
43,2.616.99
59,2.873.99


In [27]:
# share of order lines with 2 dots, in %
((orderlines_s3["unit_price"].str.count(r"\.") > 1).mean() * 100).round(2)

np.float64(12.3)

**What we see:**
- 36,169 prices (12.3% of order lines) have two dots, like `1.137.99`. Python can't read these as numbers.
- They look like prices over €1,000 written with a dot as a thousands separator.

**Decision (the team's agreed method):** remove the **whole order** whenever one of its lines has a two-dot price. If we removed only the broken line, the order's other products would stay and the order's total would no longer add up.

In [28]:
# the orders that contain at least one two-dot price
two_dot_mask = orderlines_s3["unit_price"].str.count(r"\.") > 1
orders_with_two_dots = orderlines_s3.loc[two_dot_mask, "id_order"]
orders_with_two_dots.nunique()

34642

In [29]:
# keep only the lines of orders without any two-dot price
orderlines_s3 = orderlines_s3.loc[~orderlines_s3["id_order"].isin(orders_with_two_dots), :]
orderlines_s3.shape

(216250, 7)

In [30]:
# now every price can be turned into a number
orderlines_s3["unit_price"] = pd.to_numeric(orderlines_s3["unit_price"])
orderlines_s3["unit_price"].describe()

,unit_price
count,216250.00
mean,164.64
std,219.63
min,-119.00
25%,29.74
50%,69.99
75%,184.58
max,999.99


**What we see:**
- 34,642 orders had a two-dot price. Removing them takes away 77,733 order lines: 216,250 lines are left.
- `unit_price` is now a number.
- The lowest price is −€119, and some prices are €0. These are not technical errors, so they are checked in `02_data-quality.ipynb`.
- **Caveat:** the highest price left is €999.99. Because the two-dot prices were the ones over €1,000, removing them also removes every sale of €1,000 or more. Expensive products (for example Macs) are therefore under-represented in all later analysis.

### Products: `price`

In [31]:
# split each price at the dots, to find the broken ones
split_price = products_s3["price"].str.split(".")

# broken type 1: more than one dot (more than 2 pieces)
more_than_one_dot = split_price.str.len() > 2

# broken type 2: one dot, but more than 2 digits after it
three_decimal_places = (split_price.str.len() == 2) & (split_price.str[-1].str.len() > 2)

broken_price_mask = more_than_one_dot | three_decimal_places
more_than_one_dot.sum(), three_decimal_places.sum(), broken_price_mask.sum()

(np.int64(377), np.int64(165), np.int64(542))

In [32]:
# a few of the broken prices
products_s3.loc[broken_price_mask, ["sku", "name", "price"]].head()

,sku,name,price
362,REP0043,Speaker lower repair iPhone 4,499.004
480,PIE0011,Internal Battery for iPhone 3G,98.978
515,SEN0061,Sennheiser EZX 80 Handsfree iPhone iPad and iP...,649.891
518,SEV0026,Service installation RAM + HDD + SSD MacBook /...,599.918
525,SEV0024,Service installation RAM + HDD + SSD Mac mini,599.918


In [33]:
# remove the products with a broken price, then turn price into a number
products_s3 = products_s3.loc[~broken_price_mask, :]
products_s3["price"] = products_s3["price"].astype(float)
products_s3.shape

(9992, 7)

In [34]:
# smallest and largest price, and products still without a type
products_s3["price"].min(), products_s3["price"].max(), products_s3["type"].isna().sum()

(2.99, 15339.0, np.int64(46))

**What we see:**
- 542 product prices are broken: 377 have more than one dot and 165 have three or more decimal places. These are removed.
- 9,992 products are left, and `price` is now a number.
- 46 of the 50 products without a `type` are still in the table (the other 4 had a broken price).

## Step 4: Unusable column

In [35]:
products_s4 = products_s3.copy()

In [36]:
# the promo_price column, checked with the same broken-price test
split_promo = products_s4["promo_price"].str.split(".")
broken_promo_mask = (split_promo.str.len() > 2) | ((split_promo.str.len() == 2) & (split_promo.str[-1].str.len() > 2))
broken_promo_mask.sum(), products_s4.shape[0]

(np.int64(9232), 9992)

In [37]:
# remove the column
products_s4 = products_s4.drop(columns=["promo_price"])
products_s4.columns

Index(['sku', 'name', 'desc', 'price', 'in_stock', 'type'], dtype='object')

**What we see:** 9,232 of the 9,992 promo prices (92%) are broken. The column can't be trusted, so it is removed.

## Step 5: Final check and export

In [38]:
# the last version of each table (step 4 only changed products)
orders_cl = orders_s3.copy()
orderlines_cl = orderlines_s3.copy()
products_cl = products_s4.copy()

In [39]:
# final size of each table
orders_cl.shape, orderlines_cl.shape, products_cl.shape

((226904, 4), (216250, 7), (9992, 6))

In [40]:
orders_cl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 226904 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226904 non-null  int64         
 1   created_date  226904 non-null  datetime64[ns]
 2   total_paid    226904 non-null  float64       
 3   state         226904 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 8.7+ MB


In [41]:
orderlines_cl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 216250 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                216250 non-null  int64         
 1   id_order          216250 non-null  int64         
 2   product_id        216250 non-null  int64         
 3   product_quantity  216250 non-null  int64         
 4   sku               216250 non-null  object        
 5   unit_price        216250 non-null  float64       
 6   date              216250 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(4), object(1)
memory usage: 13.2+ MB


In [42]:
products_cl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       9992 non-null   object 
 1   name      9992 non-null   object 
 2   desc      9992 non-null   object 
 3   price     9992 non-null   float64
 4   in_stock  9992 non-null   int64  
 5   type      9946 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 546.4+ KB


**What we see:**

| Table | Raw rows | Clean rows | Removed |
|---|---|---|---|
| orders | 226,909 | 226,904 | 5 with no `total_paid` |
| orderlines | 293,983 | 216,250 | 77,733 lines from 34,642 orders with a two-dot price |
| products | 19,326 | 9,992 | 8,746 duplicates, 46 with no price, 542 with a broken price |

- Dates are dates and prices are numbers.
- The only missing values left are the 46 products without a `type` (kept on purpose).
- `orderlines` still has the `product_id` column, which is always 0. It is removed in `02_data-quality.ipynb`.

In [43]:
# save the clean tables as CSV files
orders_cl.to_csv("orders_cl.csv", index=False)
orderlines_cl.to_csv("orderlines_cl.csv", index=False)
products_cl.to_csv("products_cl.csv", index=False)

In [44]:
# download them from Colab (then upload to data/clean-data)
from google.colab import files
files.download("orders_cl.csv")
files.download("orderlines_cl.csv")
files.download("products_cl.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>